# Lab 7: Pipelines and Hyperparameter Tuning

For this exercise, we will be using the titanic dataset from Lab 6. This dataset uses information about each passenger to predict if they survived or not.

In [1]:
# import the libraries we need
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
import warnings
warnings.filterwarnings('ignore') #ignoring some deprication warnings

### Dataset

You can download the dataset from Kaggle ([link](https://www.kaggle.com/competitions/titanic/data?select=train.csv)) and use the train.csv or download the file from D2L.

In [3]:
# Load the Titanic dataset into titanic_df
titanic_df = pd.read_csv('titanic.csv')
titanic_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,Mr. Owen Harris Braund,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cu...,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,Miss. Laina Heikkinen,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,Mr. William Henry Allen,male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,Rev. Juozas Montvila,male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,Miss. Margaret Edith Graham,female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Miss. Catherine Helen """"Carrie"""" Johnston",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,Mr. Karl Howell Behr,male,26.0,0,0,111369,30.0000,C148,C


In [4]:
# Get a summary of the dataset with .info
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
# Summary statistics for numerical columns
titanic_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
# Count the number of missing values in each column
titanic_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
# Remove any uniformative features and save the new dataset into titanic_df
titanic_df = titanic_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# drop all nulls
titanic_df.dropna(inplace=True)

In [8]:
# check null values and data types
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Sex       712 non-null    object 
 3   Age       712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  712 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 50.1+ KB


In [9]:
# Create column transformer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ct = ColumnTransformer(
    [("scaling", StandardScaler(), make_column_selector(dtype_exclude=object)),
     ("onehot", OneHotEncoder(sparse_output=False), make_column_selector(dtype_include=object))])

In [10]:
# Build the pipeline with Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[('preprocessor', ct),
                      ('classifier', LogisticRegression())])

In [11]:
# Split data into features (X) and target (y)
X = titanic_df.drop('Survived', axis=1)
y = titanic_df['Survived']

In [12]:
# Split the data into training and testing sets. Use 10% for testing and random_state=0
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

In [13]:
# Fit pipe to data and print training accuracy
pipe.fit(X_train, y_train)

print("Training score: {:.2f}".format(pipe.score(X_train, y_train)))

Training score: 0.80


In [14]:
# Create parameter grid with suitable values - Compare Logistic Regression to SVC
from sklearn.svm import SVC

param_grid = [{'classifier': [LogisticRegression()], 
               'classifier__C': [0.01, 0.1, 1.0, 10.0],
               'classifier__fit_intercept': [True, False]
              },
              {'classifier': [SVC()], 
               'classifier__C': [0.01, 0.1, 1.0, 10.0],
               'classifier__gamma': [0.01, 0.1, 1.0, 10.0]
              }]

In [15]:
# Initialize grid search using pipeline and parameter grid
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe, param_grid, cv=5)

In [16]:
# Fit the grid to the training data
grid.fit(X_train, y_train)

# Print out best parameters, best cross-validation score and the test score
print("Best params:\n{}\n".format(grid.best_params_))
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))

Best params:
{'classifier': SVC(gamma=0.1), 'classifier__C': 1.0, 'classifier__gamma': 0.1}

Best cross-validation score: 0.83
Test-set score: 0.75
